# Fase 2 · Perfilado y calidad de datos

Recorrido por las 9 tablas de Olist para encontrar problemas **antes** de construir el ETL. Cada hallazgo tiene un código (Q01, Q02…) que se documenta con su decisión en [docs/02_calidad_datos.md](../docs/02_calidad_datos.md) y se convierte en una regla de validación en la Fase 3.

Las funciones de chequeo están en `src/profiling.py`, con tests, no en el notebook: así se prueban con datos armados a mano antes de confiar en lo que muestran.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # para importar src/ desde notebooks/

import pandas as pd

from src.paths import RAW_DIR
from src.profiling import (fechas_invertidas, huerfanos, perfil_columnas,
                           resumen_tabla, variantes_de_texto)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

## 1. Carga

Todo se lee **como texto** primero. Si pandas infiere los tipos, los códigos postales se convierten en números y pierden el cero inicial (ver Q20). Los tipos se asignan después, a propósito.

In [2]:
ARCHIVOS = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "translation": "product_category_name_translation.csv",
    "geolocation": "olist_geolocation_dataset.csv",
}
t = {nombre: pd.read_csv(RAW_DIR / archivo, dtype=str) for nombre, archivo in ARCHIVOS.items()}

FECHAS = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
NUMEROS = {
    "items": ["order_item_id", "price", "freight_value"],
    "payments": ["payment_sequential", "payment_installments", "payment_value"],
    "reviews": ["review_score"],
    "products": ["product_name_lenght", "product_description_lenght", "product_photos_qty",
                 "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"],
    "geolocation": ["geolocation_lat", "geolocation_lng"],
}
for tabla, cols in FECHAS.items():
    t[tabla][cols] = t[tabla][cols].apply(pd.to_datetime)
for tabla, cols in NUMEROS.items():
    t[tabla][cols] = t[tabla][cols].apply(pd.to_numeric)

orders, items, payments, reviews = t["orders"], t["items"], t["payments"], t["reviews"]
customers, sellers, products, translation, geo = (t["customers"], t["sellers"], t["products"],
                                                  t["translation"], t["geolocation"])

## 2. Panorama: tamaño, duplicados y claves

La **clave** es la combinación de columnas que debería identificar cada fila. Si se repite, la tabla no tiene la granularidad que creemos.

In [3]:
CLAVES = {
    "orders": ["order_id"], "items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"], "reviews": ["review_id"],
    "customers": ["customer_id"], "sellers": ["seller_id"], "products": ["product_id"],
    "translation": ["product_category_name"], "geolocation": ["geolocation_zip_code_prefix"],
}
pd.DataFrame({nombre: resumen_tabla(t[nombre], clave) for nombre, clave in CLAVES.items()}).T

,filas,columnas,filas duplicadas,claves repetidas,clave única
orders,99441,8,0,0,True
items,112650,7,0,0,True
payments,103886,5,0,0,True
reviews,99224,7,0,814,False
customers,99441,5,0,0,True
sellers,3095,4,0,0,True
products,32951,9,0,0,True
translation,71,2,0,0,True
geolocation,1000163,5,261831,981148,False


Dos tablas no cumplen su clave:

- **Q04 · `reviews`:** `review_id` se repite. Más abajo se ve que una misma reseña aparece asociada a varios pedidos.
- **Q01 · `geolocation`:** no hay una fila por código postal sino ~29 en promedio, y 261.831 filas son copias exactas.

## 3. Perfil de columnas

Nulos, valores distintos y rangos de cada columna. Se muestran las tablas con algo para notar.

In [4]:
perfil_columnas(orders)

,tipo,nulos,% nulos,distintos,mínimo,máximo,ejemplo
columna,,,,,,,
order_id,str,0,0.00,99441,NaT,NaT,e481f51cbdc54678b7cc49136f2d6af7
customer_id,str,0,0.00,99441,NaT,NaT,9ef432eb6251297304e76186b10a928d
order_status,str,0,0.00,8,NaT,NaT,delivered
order_purchase_timestamp,datetime64[us],0,0.00,98875,2016-09-04 21:15:19,2018-10-17 17:30:18,2017-10-02 10:56:33
order_approved_at,datetime64[us],160,0.16,90733,2016-09-15 12:16:38,2018-09-03 17:40:06,2017-10-02 11:07:15
order_delivered_carrier_date,datetime64[us],1783,1.79,81018,2016-10-08 10:34:01,2018-09-11 19:48:28,2017-10-04 19:55:00
order_delivered_customer_date,datetime64[us],2965,2.98,95664,2016-10-11 13:46:32,2018-10-17 13:22:46,2017-10-10 21:25:13
order_estimated_delivery_date,datetime64[us],0,0.00,459,2016-09-30 00:00:00,2018-11-12 00:00:00,2017-10-18 00:00:00


Los nulos de `orders` son esperables en principio: un pedido que no se aprobó no tiene fecha de aprobación, y uno que no se entregó no tiene fecha de entrega. En la sección 6 se verifica que los nulos coincidan con el estado.

In [5]:
perfil_columnas(items)

,tipo,nulos,% nulos,distintos,mínimo,máximo,ejemplo
columna,,,,,,,
order_id,str,0,0.0,98666,None,None,00010242fe8c5a6d1ba2dd792cb16214
order_item_id,int64,0,0.0,21,1,21,1
product_id,str,0,0.0,32951,None,None,4244733e06e7ecb4970a6e2683c13e61
seller_id,str,0,0.0,3095,None,None,48436dade18ac8b2bce089ec2a041202
shipping_limit_date,datetime64[us],0,0.0,93318,2016-09-19 00:15:34,2020-04-09 22:35:08,2017-09-19 09:45:35
price,float64,0,0.0,5968,0.85,6735.0,58.9
freight_value,float64,0,0.0,6999,0.0,409.68,13.29


- `price` nunca es 0 ni negativo. `freight_value` es 0 en algunos ítems (**Q16**, envío gratis).
- `shipping_limit_date` llega hasta **2020**, cuando los pedidos terminan en 2018 (**Q10**).

In [6]:
perfil_columnas(products)

,tipo,nulos,% nulos,distintos,mínimo,máximo,ejemplo
columna,,,,,,,
product_id,str,0,0.00,32951,NaN,NaN,1e9e8ef04dbcff4541ed26657ea517e5
product_category_name,str,610,1.85,73,NaN,NaN,perfumaria
product_name_lenght,float64,610,1.85,66,5.0,76.0,40.0
product_description_lenght,float64,610,1.85,2960,4.0,3992.0,287.0
product_photos_qty,float64,610,1.85,19,1.0,20.0,1.0
product_weight_g,float64,2,0.01,2204,0.0,40425.0,225.0
product_length_cm,float64,2,0.01,99,7.0,105.0,16.0
product_height_cm,float64,2,0.01,102,2.0,105.0,10.0
product_width_cm,float64,2,0.01,95,6.0,118.0,14.0


- 610 productos sin categoría y sin datos de nombre, descripción y fotos (**Q17**).
- Dos columnas tienen el nombre mal escrito en el origen: `lenght` en vez de `length` (**Q19**).

In [7]:
perfil_columnas(reviews)

,tipo,nulos,% nulos,distintos,mínimo,máximo,ejemplo
columna,,,,,,,
review_id,str,0,0.00,98410,None,None,7bc2406110b926393aa56f80a40eba40
order_id,str,0,0.00,98673,None,None,73fc7af87114b39712e6da79b0a377eb
review_score,int64,0,0.00,5,1,5,4
review_comment_title,str,87656,88.34,4527,None,None,recomendo
review_comment_message,str,58247,58.70,36159,None,None,Recebi bem antes do prazo estipulado.
review_creation_date,datetime64[us],0,0.00,636,2016-10-02 00:00:00,2018-08-31 00:00:00,2018-01-18 00:00:00
review_answer_timestamp,datetime64[us],0,0.00,98248,2016-10-07 18:32:28,2018-10-29 12:27:35,2018-01-18 21:46:59


Los comentarios están vacíos en el 88% (título) y 59% (mensaje) de las reseñas (**Q22**).

## 4. Integridad referencial

¿Cada clave foránea apunta a una fila que existe? Un huérfano rompe las relaciones del modelo en estrella y hace que los totales no cierren.

In [8]:
RELACIONES = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("items", "order_id", "orders", "order_id"),
    ("items", "product_id", "products", "product_id"),
    ("items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("products", "product_category_name", "translation", "product_category_name"),
    ("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
]
pd.DataFrame(
    [(f"{h}.{ch}", f"{p}.{cp}", huerfanos(t[h], ch, t[p], cp)) for h, ch, p, cp in RELACIONES],
    columns=["hijo", "padre", "huérfanos"],
)

,hijo,padre,huérfanos
0,orders.customer_id,customers.customer_id,0
1,items.order_id,orders.order_id,0
2,items.product_id,products.product_id,0
3,items.seller_id,sellers.seller_id,0
4,payments.order_id,orders.order_id,0
5,reviews.order_id,orders.order_id,0
6,products.product_category_name,translation.product_category_name,13
7,customers.customer_zip_code_prefix,geolocation.geolocation_zip_code_prefix,278
8,sellers.seller_zip_code_prefix,geolocation.geolocation_zip_code_prefix,7


Las relaciones del núcleo transaccional (pedidos, ítems, pagos, reseñas, clientes, vendedores, productos) están **completas**. Los huérfanos aparecen en las tablas auxiliares:

- **Q18:** 2 categorías sin traducción.
- **Q02:** códigos postales de clientes y vendedores que no están en `geolocation`.

¿Y al revés, pedidos sin pagos o sin ítems?

In [9]:
pd.Series({
    "pedidos sin pago": int((~orders.order_id.isin(payments.order_id)).sum()),
    "pedidos sin ítems": int((~orders.order_id.isin(items.order_id)).sum()),
    "pedidos sin reseña": int((~orders.order_id.isin(reviews.order_id)).sum()),
})

pedidos sin pago        1
pedidos sin ítems     775
pedidos sin reseña    768
dtype: int64

In [10]:
orders[~orders.order_id.isin(items.order_id)].order_status.value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Los 775 pedidos sin ítems son casi todos `unavailable` o `canceled`, que no cuentan como venta (regla R1). Los que quedan en estados intermedios ya están contemplados en la regla R8. El pedido sin pago (**Q15**) es uno solo.

## 5. Granularidad: la trampa de unir ítems con pagos

Un pedido puede tener **varios ítems** y **varios pagos**. Son dos relaciones uno a muchos que salen del mismo pedido.

In [11]:
pd.DataFrame({
    "ítems por pedido": items.groupby("order_id").size().describe(),
    "pagos por pedido": payments.groupby("order_id").size().describe(),
    "vendedores por pedido": items.groupby("order_id").seller_id.nunique().describe(),
    "reseñas por pedido": reviews.groupby("order_id").size().describe(),
}).round(2)

,ítems por pedido,pagos por pedido,vendedores por pedido,reseñas por pedido
count,98666.00,99440.00,98666.00,98673.00
mean,1.14,1.04,1.01,1.01
std,0.54,0.38,0.12,0.08
min,1.00,1.00,1.00,1.00
25%,1.00,1.00,1.00,1.00
50%,1.00,1.00,1.00,1.00
75%,1.00,1.00,1.00,1.00
max,21.00,29.00,5.00,3.00


Si se unen ítems y pagos directamente, cada ítem se repite por cada pago y cada pago por cada ítem (**Q12**). El error no da ningún aviso: el número simplemente sale más grande.

In [12]:
correcto = items.price.sum()
unido = items.merge(payments, on="order_id").price.sum()
print(f"Ventas correctas:              R$ {correcto:,.2f}")
print(f"Ventas tras unir con pagos:    R$ {unido:,.2f}")
print(f"Inflado:                       {unido / correcto - 1:+.1%}")

Ventas correctas:              R$ 13,591,643.70
Ventas tras unir con pagos:    R$ 14,209,115.34
Inflado:                       +4.5%


Por eso el modelo tiene **dos tablas de hechos con distinta granularidad** (ítem y pedido) y los pagos se agregan por pedido antes de unirse con nada.

También hay pedidos con **más de un vendedor** (**Q13**):

In [13]:
vendedores = items.groupby("order_id").seller_id.nunique()
print(f"Pedidos con más de un vendedor: {(vendedores > 1).sum():,} ({(vendedores > 1).mean():.1%})")

Pedidos con más de un vendedor: 1,278 (1.3%)


## 6. Fechas

### Rango y meses incompletos

In [14]:
orders.groupby(orders.order_purchase_timestamp.dt.to_period('M')).size().to_frame('pedidos').T

order_purchase_timestamp,2016-09,2016-10,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,...,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10
pedidos,4,324,1,800,1780,2682,2404,3700,3245,4026,...,7269,6728,7211,6939,6873,6167,6292,6512,16,4


Los meses de 2016 y septiembre–octubre de 2018 están casi vacíos: es el motivo de la decisión D-008 (período ene-2017 a ago-2018).

### Secuencia lógica

Compra → aprobación → despacho → entrega. Ninguna fecha debería ser anterior a la del paso previo.

In [15]:
PASOS = [
    ("order_purchase_timestamp", "order_approved_at"),
    ("order_purchase_timestamp", "order_delivered_carrier_date"),
    ("order_approved_at", "order_delivered_carrier_date"),
    ("order_delivered_carrier_date", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_estimated_delivery_date"),
]
pd.DataFrame(
    [(a, p, fechas_invertidas(orders, a, p)) for a, p in PASOS],
    columns=["debería ser antes", "debería ser después", "filas invertidas"],
)

,debería ser antes,debería ser después,filas invertidas
0,order_purchase_timestamp,order_approved_at,0
1,order_purchase_timestamp,order_delivered_carrier_date,166
2,order_approved_at,order_delivered_carrier_date,1359
3,order_delivered_carrier_date,order_delivered_customer_date,23
4,order_purchase_timestamp,order_delivered_customer_date,0
5,order_purchase_timestamp,order_estimated_delivery_date,0


**Q09:** hay pasos intermedios desordenados (despacho antes de la aprobación, entrega antes del despacho). Pero las tres fechas que usan los KPIs —**compra, entrega y entrega estimada**— están siempre en orden. Eso define qué regla corta la carga y cuál solo avisa.

In [16]:
dias = (orders.order_delivered_customer_date - orders.order_purchase_timestamp).dt.total_seconds() / 86400
dias.describe(percentiles=[.5, .9, .99]).round(1)

count    96476.0
mean        12.6
std          9.5
min          0.5
50%         10.2
90%         23.1
99%         46.0
max        209.6
dtype: float64

**Q11:** entregas de menos de un día y de más de 200 días. Son extremos pero posibles; por eso el diccionario muestra la mediana junto al promedio (K18).

### Estado del pedido contra las fechas cargadas

In [17]:
orders.assign(
    aprobado=orders.order_approved_at.notna(),
    despachado=orders.order_delivered_carrier_date.notna(),
    entregado=orders.order_delivered_customer_date.notna(),
).groupby("order_status")[["aprobado", "despachado", "entregado"]].sum().assign(
    total=orders.order_status.value_counts())

,aprobado,despachado,entregado,total
order_status,,,,
approved,2,0,0,2
canceled,484,75,6,625
created,0,0,0,5
delivered,96464,96476,96470,96478
invoiced,314,0,0,314
processing,301,0,0,301
shipped,1107,1107,0,1107
unavailable,609,0,0,609


- **Q06:** 8 pedidos `delivered` sin fecha de entrega.
- **Q07:** 14 pedidos `delivered` sin fecha de aprobación.
- **Q08:** 6 pedidos `canceled` que tienen fecha de entrega.

## 7. Reseñas

In [18]:
por_pedido = reviews.groupby("order_id").agg(reseñas=("review_id", "size"), puntajes=("review_score", "nunique"))
pd.Series({
    "review_id repetidos": int(reviews.review_id.duplicated().sum()),
    "pedidos por review_id (máximo)": int(reviews.groupby("review_id").order_id.nunique().max()),
    "pedidos con más de una reseña": int((por_pedido.reseñas > 1).sum()),
    "  ...con puntajes distintos": int((por_pedido.puntajes > 1).sum()),
})

review_id repetidos               814
pedidos por review_id (máximo)      3
pedidos con más de una reseña     547
  ...con puntajes distintos       202
dtype: int64

**Q04:** una misma reseña aparece en hasta 3 pedidos (probablemente pedidos comprados juntos), y 547 pedidos tienen más de una reseña, 202 con puntajes distintos. La regla R7 resuelve cuál vale: la última respondida.

In [19]:
r = reviews.merge(orders[["order_id", "order_purchase_timestamp", "order_delivered_customer_date"]], on="order_id")
pd.Series({
    "reseña creada antes de la compra": int((r.review_creation_date < r.order_purchase_timestamp.dt.normalize()).sum()),
    "reseña creada antes de la entrega": int((r.review_creation_date < r.order_delivered_customer_date.dt.normalize()).sum()),
})

reseña creada antes de la compra       64
reseña creada antes de la entrega    5127
dtype: int64

**Q05:** que una reseña se cree antes de la entrega es esperable: Olist envía la encuesta en la fecha estimada, llegue o no el pedido. Que se cree **antes de la compra** es un error de datos.

## 8. Pagos

In [20]:
total_pagado = payments.groupby("order_id").payment_value.sum()
total_items = items.assign(total=items.price + items.freight_value).groupby("order_id").total.sum()
diferencia = (total_pagado - total_items).dropna()
pd.Series({
    "pedidos con diferencia > R$ 1": int((diferencia.abs() > 1).sum()),
    "  ...se pagó de más": int((diferencia > 1).sum()),
    "  ...se pagó de menos": int((diferencia < -1).sum()),
    "pagos con valor 0": int((payments.payment_value <= 0).sum()),
    "pagos tipo not_defined": int((payments.payment_type == "not_defined").sum()),
    "pagos con 0 cuotas": int((payments.payment_installments == 0).sum()),
})

pedidos con diferencia > R$ 1    249
  ...se pagó de más              232
  ...se pagó de menos             17
pagos con valor 0                  9
pagos tipo not_defined             3
pagos con 0 cuotas                 2
dtype: int64

**Q14:** en 249 pedidos lo pagado no coincide con productos + flete; en la mayoría se pagó de más (posiblemente intereses por cuotas). **Q15:** pagos en 0, tipo `not_defined` (todos de pedidos cancelados) y 0 cuotas.

## 9. Productos y categorías

In [21]:
sin_traduccion = products.product_category_name.dropna()
sin_traduccion = sin_traduccion[~sin_traduccion.isin(translation.product_category_name)]
print("Categorías sin traducción:", sin_traduccion.value_counts().to_dict())
print("Productos sin categoría:", products.product_category_name.isna().sum())
print("Productos con peso 0:", (products.product_weight_g == 0).sum())

Categorías sin traducción: {'portateis_cozinha_e_preparadores_de_alimentos': 10, 'pc_gamer': 3}
Productos sin categoría: 610
Productos con peso 0: 4


## 10. Geolocalización

In [22]:
por_zip = geo.groupby("geolocation_zip_code_prefix").agg(
    filas=("geolocation_lat", "size"), estados=("geolocation_state", "nunique"))
# Rectángulo aproximado de Brasil: una fila cuenta una vez aunque falle latitud y longitud
fuera_brasil = int((~geo.geolocation_lat.between(-33.8, 5.3) | ~geo.geolocation_lng.between(-74.0, -34.7)).sum())
pd.Series({
    "filas": len(geo),
    "copias exactas": int(geo.duplicated().sum()),
    "códigos postales distintos": len(por_zip),
    "filas por código postal (mediana)": int(por_zip.filas.median()),
    "filas por código postal (máximo)": int(por_zip.filas.max()),
    "códigos postales con más de un estado": int((por_zip.estados > 1).sum()),
    "coordenadas fuera de Brasil (aprox.)": fuera_brasil,
})

filas                                    1000163
copias exactas                            261831
códigos postales distintos                 19015
filas por código postal (mediana)             29
filas por código postal (máximo)            1146
códigos postales con más de un estado          8
coordenadas fuera de Brasil (aprox.)          42
dtype: int64

**Q01:** la tabla no es un catálogo de códigos postales sino una lista de puntos. Para el modelo hace falta **una fila por código postal**. Además hay coordenadas fuera de Brasil (aprox., por latitud o longitud fuera del rectángulo del país).

## 11. Textos: ciudades

In [23]:
variantes_de_texto(geo.geolocation_city).head(8)

,variantes,formas
normalizado,,
ipora,3,"[ipora, iporá, iporã]"
arapua,3,"[arapua, arapuá, arapuã]"
barra de santo antonio,3,"[barra de santo antônio, barra de santo anton..."
santa barbara d'oeste,3,"[santa barbara d'oeste, santa bárbara d'oeste,..."
santo estevao,3,"[santo estevao, santo estevão, santo estêvão]"
sao paulo,3,"[sao paulo, sa£o paulo, são paulo]"
sao sebastiao do passe,3,"[sao sebastiao do passe, são sebastião do pass..."
teresopolis,3,"[teresopolis, teresópolis, ´teresopolis]"


In [24]:
sucias = sellers[sellers.seller_city.str.contains(r"[0-9@/\\,]| - |^sp$", regex=True)]
print(f"Ciudades de vendedores con formato inválido: {len(sucias)}")
sucias[["seller_city", "seller_state"]].head(10)

Ciudades de vendedores con formato inválido: 28


,seller_city,seller_state
78,lages - sc,SC
237,auriflama/sp,SP
246,sao paulo / sao paulo,SP
517,04482255,RJ
551,"novo hamburgo, rio grande do sul, brasil",RS
622,cariacica / es,ES
826,sao paulo - sp,SP
869,sbc/sp,SP
945,santo andre/sao paulo,SP
1004,sp / sp,SP


**Q03:** la misma ciudad escrita con y sin tilde ("são paulo" / "sao paulo"), con caracteres corruptos por un problema de codificación ("sa£o paulo", "´teresopolis"), y ciudades de vendedores cargadas como "sao paulo - sp", "sbc/sp" o directamente un número. La ciudad no se usa en ningún KPI (la región es el estado, regla R9), así que se normaliza pero no bloquea nada.

## 12. Códigos postales: el cero que desaparece

In [25]:
como_texto = customers.customer_zip_code_prefix.str.len().value_counts()
como_numero = pd.read_csv(RAW_DIR / ARCHIVOS["customers"]).customer_zip_code_prefix.astype(str).str.len().value_counts()
pd.DataFrame({"leído como texto": como_texto, "leído como número": como_numero}).rename_axis("dígitos")

,leído como texto,leído como número
dígitos,,
4,NaN,23995
5,99441.0,75446


**Q20:** leído como número, **uno de cada cuatro** códigos postales pierde el cero inicial (`09790` → `9790`) y deja de coincidir con `geolocation`. No da error: la unión simplemente no encuentra la fila. En SQL Server la columna va como `CHAR(5)`.

---

## Resumen

Todos los hallazgos, su impacto y la decisión tomada están en [docs/02_calidad_datos.md](../docs/02_calidad_datos.md).